In [1]:
from gam_rs_utils.utils import *
from method_scripts.results import Results

In [2]:
dn = 'bank'
data = pd.read_csv(f'datasets/{dn}.csv')
l0 = 0.001
l2 = 0.001
m = 1.01

fastsparse_data = Results.create_fastsparse_dataset(dn, l0, l2)
bin_X = fastsparse_data['bin_X']
cum_X = fastsparse_data['cum_X']
cum_header = fastsparse_data['cum_header']
bin_X = fastsparse_data['bin_X']
bin_header = fastsparse_data['bin_header']
y = fastsparse_data['y']
w = fastsparse_data['w']

print("data shape", data.shape)
print("w", len(w), "y", len(y), "header", len(cum_header))
print("cum_X shape", cum_X.shape)

loading dataset from results/bank/l0_0.001_l2_0.001.pkl
Loading cached fastsparse dataset for bank with l0=0.001 and l2=0.001
data shape (4521, 17)
w 3719 y 4521 header 3719
cum_X shape (4521, 3719)


In [3]:
from FasterRisk.src.fasterrisk import fasterrisk
from time import time

start = time()

rs = fasterrisk.RiskScoreOptimizer(
    cum_X[:,1:], y, 
    k=w.nonzero()[0].shape[0]-1, 
    lb=-100, ub=100, 
    gap_tolerance=m - 1.0, 
    select_top_m=-1, 
    maxAttempts=25
)
beam_size = 100
rs.optimize_with_swaps_beam_search(
    swaps=2,
    beam_size=beam_size, 
    verbose=True, 
    beta0=w[0],
    betas=w[1:]
)

end = time()

swap 0, beam size 1
swap 1, beam size 46


KeyboardInterrupt: 

In [ ]:
w_opt = np.concatenate([np.array([rs.opt_beta0]), rs.opt_betas])
w_rset = np.column_stack([rs.sparseDiversePool_beta0, rs.sparseDiversePool_betas])
l2 = rs.lambda2
rset_bound = rs.rset_bound

In [ ]:
ModelUtils.print_results_summary(
    w_rset, w_opt, 
    cum_X, y, l2, 
    end - start
)

	1595 solutions, 145.72 seconds
Average logistic loss:  0.32876385754000204
Opt model logistic loss:  3.3957904477958825
